# Fake News Detection using BERT

## Mini Project – Transformers & Modern NLP

### Objective
Build a fake news classification system using a pretrained transformer model and fine-tune it on a real dataset.

### Model Used
- bert-base-uncased
- distilbert-base-uncased

### Libraries
- HuggingFace Transformers
- PyTorch
- Scikit-learn
- Pandas
- Seaborn
- Gradio

### Author
Aryan Eshwar K R

In [ ]:
!pip install -U transformers

In [ ]:
!pip install transformers --upgrade

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
!pip install transformers datasets scikit-learn pandas matplotlib seaborn

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
import pandas as pd

fake = pd.read_csv("/content/drive/MyDrive/Fake.csv")
true = pd.read_csv("/content/drive/MyDrive/True.csv")

fake["label"] = 0
true["label"] = 1

data = pd.concat([fake, true])
data = data[["title","text","label"]]

data.head()

In [ ]:
print("Dataset Shape:", data.shape)
print(data.info())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot label distribution
sns.countplot(x=data["label"])
plt.title("Label Distribution (Fake vs Real)")
plt.xlabel("Label (0 = Fake, 1 = Real)")
plt.ylabel("Count")
plt.show()

# Print counts
print(data["label"].value_counts())

## Dataset Summary

The dataset contains **XXXX samples** of news articles/headlines used for fake news classification.

### Labels:
- **0 → Fake News**
- **1 → Real News**

### Class Distribution:
- Fake News: 23481 samples  
- Real News: 21417 samples  

### Observation:
The dataset is (balanced / slightly imbalanced).  
This can impact model performance, as models may become biased toward the majority class.

In [ ]:
data.sample(5)

## Tokenization & Data Preparation

Transformer models like BERT cannot understand raw text directly.  
So we convert text into numerical tokens using a tokenizer.

We use:
- **BERT Tokenizer (bert-base-uncased)**
- Padding → ensures equal length inputs
- Truncation → limits long sentences
- Max length → 256 tokens

We then split the dataset into training and validation sets and convert it into PyTorch format.

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
tokens = tokenizer(
    data["text"].tolist(),
    padding=True,
    truncation=True,
    max_length=256
)

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    data["text"],
    data["label"],
    test_size=0.2,
    random_state=42
)

In [ ]:
import torch

train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
print("Sample Text:\n", train_texts[0])

print("\nToken IDs:\n", train_encodings["input_ids"][0])

print("\nAttention Mask:\n", train_encodings["attention_mask"][0])

In [ ]:
print("Train Input Shape:", len(train_encodings["input_ids"]), "x", len(train_encodings["input_ids"][0]))
print("Validation Input Shape:", len(val_encodings["input_ids"]), "x", len(val_encodings["input_ids"][0]))

In [ ]:
class FakeNewsDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = FakeNewsDataset(train_encodings, train_labels)
val_dataset = FakeNewsDataset(val_encodings, val_labels)

### Observations

- Each sentence is converted into **input_ids** (token numbers)
- **Attention mask** tells the model which tokens are real vs padding
- All sequences are padded to length **256**
- Data is now ready for training using PyTorch

This step is crucial for feeding text into transformer models.

In [ ]:
print("Decoded back:", tokenizer.decode(train_encodings["input_ids"][0]))

In [ ]:
train_dataset[0]

## Model Fine-Tuning

In this step, we fine-tune a pretrained **BERT model** for fake news classification.

Why fine-tuning?
- BERT is already trained on large text corpora
- We adapt it to our specific task (Fake vs Real classification)

We use:
- **bert-base-uncased**
- 2–3 epochs for training
- Evaluation at each epoch

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [ ]:
from transformers import Trainer
from sklearn.metrics import accuracy_score

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)

    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/content/drive/MyDrive/fake_news_model")

### Training Observations

- The model was trained for 3 epochs
- Training loss decreased over time
- Validation metrics improved, indicating learning

Metrics tracked:
- Accuracy
- Precision
- Recall
- F1-score

The model successfully learned patterns distinguishing fake and real news.

## Model Evaluation

In this section, we evaluate the performance of our fine-tuned BERT model using standard classification metrics:

- Accuracy
- Precision
- Recall
- F1-score

We also visualize results using a confusion matrix.

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

predictions = trainer.predict(val_dataset)

y_pred = predictions.predictions.argmax(-1)
y_true = val_labels

In [ ]:
import numpy as np

np.save("/content/drive/MyDrive/y_pred.npy", y_pred)
np.save("/content/drive/MyDrive/y_true.npy", y_true)

In [ ]:
print(classification_report(y_true, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Fake", "Real"],
            yticklabels=["Fake", "Real"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)
print("Accuracy:", accuracy)

In [ ]:
print(data["label"].value_counts())

In [ ]:
print(data.head())

In [ ]:
data["label"] = data["label"].map({"FAKE": 0, "REAL": 1})

In [ ]:
print(data.columns)

In [ ]:
print(train_texts[0])

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    learning_rate=2e-5,   # 🔥 IMPORTANT
)

### Evaluation Interpretation

- **Accuracy** shows the overall correctness of the model.
- **Precision** indicates how many predicted fake/real labels were actually correct.
- **Recall** shows how well the model captures all actual instances.
- **F1-score** balances precision and recall.

### Confusion Matrix Analysis

- True Positives (TP): Correctly predicted real news
- True Negatives (TN): Correctly predicted fake news
- False Positives (FP): Fake news predicted as real
- False Negatives (FN): Real news predicted as fake

### Observations

- The model performs well in distinguishing fake and real news.
- Some misclassifications occur due to:
  - Ambiguous wording
  - Lack of context
  - Similar patterns in fake and real news

Overall, the model demonstrates strong performance but can be improved further.

## Error Analysis

In this section, we analyze the model's incorrect predictions to understand its weaknesses.

By examining misclassified examples, we can identify patterns and improve model performance.

In [ ]:
val_texts = list(val_texts)
val_labels = list(val_labels)

In [ ]:
y_true = list(val_labels)

In [ ]:
y_true = np.array(val_labels)

In [ ]:
wrong_predictions = []

for i in range(len(y_pred)):
    if y_pred[i] != y_true[i]:
        wrong_predictions.append({
            "text": val_texts[i],
            "actual": y_true[i],
            "predicted": y_pred[i]
        })

len(wrong_predictions)

In [ ]:
for i in range(5):
    print(f"\nExample {i+1}")
    print("Text:", wrong_predictions[i]["text"])
    print("Actual:", "Fake" if wrong_predictions[i]["actual"] == 0 else "Real")
    print("Predicted:", "Fake" if wrong_predictions[i]["predicted"] == 0 else "Real")

In [ ]:
import pandas as pd

df_wrong = pd.DataFrame(wrong_predictions[:5])
df_wrong

### Observations from Error Analysis

After analyzing incorrect predictions, the following patterns were observed:

1. **Ambiguous Headlines**
   - Some news headlines contain vague or misleading wording.
   - The model struggles to interpret unclear context.

2. **Lack of Context**
   - Short headlines without sufficient information are harder to classify.

3. **Clickbait Language**
   - Fake news often uses exaggerated phrases.
   - However, real news sometimes uses similar language, confusing the model.

4. **Sarcasm or Irony**
   - The model cannot understand sarcasm effectively.

5. **Similar Writing Style**
   - Some fake and real articles share similar linguistic patterns.

### Conclusion from Errors

The model performs well overall but struggles with:
- Context understanding
- Subtle linguistic nuances
- Ambiguity in text

These insights can guide future improvements.

## Model Improvement

To improve efficiency and compare performance, we evaluate a lighter transformer model: **DistilBERT**.

DistilBERT is:
- Smaller and faster than BERT
- Retains ~95% of BERT performance
- More efficient for deployment

We compare both models based on accuracy and F1-score.

In [ ]:
from transformers import DistilBertForSequenceClassification
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

### Results Comparison

- BERT achieved higher accuracy and F1-score.
- DistilBERT performed slightly lower but was significantly faster.

### Key Observations

- BERT is better for accuracy-focused applications.
- DistilBERT is better for real-time or resource-constrained systems.

### Conclusion

There is a trade-off between performance and efficiency:
- **BERT → Higher accuracy**
- **DistilBERT → Faster and lighter**

This comparison highlights the importance of choosing models based on use-case requirements.

In [ ]:
tokenizer.save_pretrained("/content/drive/MyDrive/fake_news_model")

In [ ]:
import os
os.listdir("/content/drive/MyDrive/fake_news_model")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from transformers import BertForSequenceClassification, AutoTokenizer

model_path = "/content/drive/MyDrive/fake_news_model"

model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

## Model Deployment (Gradio Interface)

To demonstrate our model, we build a simple web interface using Gradio.

Users can input a news headline or article, and the model predicts whether it is:
- Fake News
- Real News

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
from transformers import pipeline

# Path where the model is saved in Google Drive
model_path = "/content/drive/MyDrive/fake_news_model"

# Load trained model
classifier = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=model_path
)

def detect_news(text):
    if text.strip() == "":
        return "Please enter some news text."

    result = classifier(text)[0]

    label = result["label"]
    score = result["score"]

    if label == "LABEL_0":
        return f"🚨 Fake News\nConfidence: {score:.2%}"
    else:
        return f"✅ Real News\nConfidence: {score:.2%}"


demo = gr.Interface(
    fn=detect_news,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Paste a news headline or article here..."
    ),
    outputs="text",
    title="📰 Fake News Detection",
    description="Detect whether a news article or headline is Fake or Real using a BERT transformer model.",
    examples=[
        ["Scientists discover water on Mars"],
        ["Government secretly controls weather using satellites"]
    ]
)

demo.launch()